In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
 # 1. Data Wrangling & Preparation : Reading all required datasets
matches = pd.read_csv('fifamatches2026/matches.csv')
stages = pd.read_csv('fifamatches2026/tournament_stages.csv')
team_stats = pd.read_csv('fifamatches2026/match_team_stats.csv')

In [ ]:
# I got required dataset by merging datasets to link stage types with team match stats
matches_stage = matches.merge(stages, on='stage_id', how='inner')
df = team_stats.merge(
    matches_stage[['match_id', 'is_knockout', 'stage_name']], 
    on='match_id', 
    how='inner'
)

In [ ]:
# Then I filtered for clean fouls data
fouls_df = df[['match_id', 'team_id', 'stage_name', 'is_knockout', 'fouls']].dropna()

In [ ]:
# 2. Data Splitting & Sampling : I split the data into two groups based on whether the match was a knockout or not
group_fouls = fouls_df[fouls_df['is_knockout'] == False]['fouls']
knockout_fouls = fouls_df[fouls_df['is_knockout'] == True]['fouls']

In [ ]:
# 3. Descriptive Statistics Helper : I created a helper function to calculate descriptive statistics like mean,
# standard deviation, and confidence intervals for each group
def get_stats(series):
    n = len(series)
    mean = series.mean()
    std = series.std(ddof=1)
    se = std / np.sqrt(n)
    median = series.median()
    q25, q75 = series.quantile(0.25), series.quantile(0.75)
    iqr = q75 - q25
    return {'N': n, 'Mean': mean, 'Std': std, 'SE': se, 'Median': median, 'IQR': iqr}

g_stats = get_stats(group_fouls)
k_stats = get_stats(knockout_fouls)

In [ ]:
# 4. Inferential Statistics: 95% Confidence Intervals : I calculated the 95% confidence intervals for the mean fouls in both groups
# using the t-distribution.
ci_group = stats.t.interval(
    0.95, df=g_stats['N']-1, loc=g_stats['Mean'], scale=g_stats['SE']
)
ci_knockout = stats.t.interval(
    0.95, df=k_stats['N']-1, loc=k_stats['Mean'], scale=k_stats['SE']
)

In [ ]:
# 5. Inferential Statistics: Two-Sample Welch's t-Test : I performed a two-sample Welch's t-test to compare the mean fouls between
# knockout and non-knockout matches, which does not assume equal population variances.
t_stat, p_val = stats.ttest_ind(knockout_fouls, group_fouls, equal_var=False)

In [ ]:
# Format & Print Summary : I formatted and printed the results of the descriptive statistics, confidence intervals,
# and t-test results in a clear and organized manner.
print("=== DESCRIPTIVE STATISTICS ===")
print(f"Group Stage    : N={g_stats['N']}, Mean={g_stats['Mean']:.2f}, Std={g_stats['Std']:.2f}, SE={g_stats['SE']:.2f}, Median={g_stats['Median']:.2f}, IQR={g_stats['IQR']:.2f}")
print(f"Knockout Stage : N={k_stats['N']}, Mean={k_stats['Mean']:.2f}, Std={k_stats['Std']:.2f}, SE={k_stats['SE']:.2f}, Median={k_stats['Median']:.2f}, IQR={k_stats['IQR']:.2f}")

print("\n=== 95% CONFIDENCE INTERVALS ===")
print(f"Group Stage Mean 95% CI    : [{ci_group[0]:.2f}, {ci_group[1]:.2f}]")
print(f"Knockout Stage Mean 95% CI : [{ci_knockout[0]:.2f}, {ci_knockout[1]:.2f}]")

print("\n=== TWO-SAMPLE T-TEST RESULTS ===")
print(f"t-statistic : {t_stat:.4f}")
print(f"p-value     : {p_val:.4f}")

In [ ]:
# 6. Visualization : I created a bar plot to visualize the mean fouls per match for both groups, including error bars
# representing the 95% confidence intervals.
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(8, 6))

stages_list = ['Group Stage', 'Knockout Stage']
means = [g_stats['Mean'], k_stats['Mean']]

# Then I formatted error bars using calculated 95% CIs
yerr = [
    [g_stats['Mean'] - ci_group[0], k_stats['Mean'] - ci_knockout[0]],
    [ci_group[1] - g_stats['Mean'], ci_knockout[1] - k_stats['Mean']]
]

bars = ax.bar(stages_list, means, yerr=yerr, capsize=8, color=["#8f2b2b", "#10d902"], alpha=0.85, width=0.45, edgecolor='black', linewidth=1)

# And annotated mean values and confidence intervals inside bars
for bar, mean, ci in zip(bars, means, [ci_group, ci_knockout]):
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval/2, f"Mean: {mean:.2f}\n95% CI: [{ci[0]:.2f}, {ci[1]:.2f}]", 
            ha='center', va='center', color='white', fontweight='bold', fontsize=11)

ax.set_ylim(0, 15)
ax.set_title('Mean Fouls per Match with 95% Confidence Intervals', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Tournament Stage', fontsize=12, labelpad=10)
ax.set_ylabel('Average Fouls per Match', fontsize=12, labelpad=10)

plt.tight_layout()
plt.show()